In [98]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. 数据分析

In [99]:
# ---------------------- 1. 读取数据 ----------------------
train_df = pd.read_csv("./train.csv")
test_df = pd.read_csv("./test.csv")

train_df.head()

,Id,Address,Sold Price,Summary,Type,Year built,Heating,Cooling,Parking,Lot,...,Parking features,Tax assessed value,Annual tax amount,Listed On,Listed Price,Last Sold On,Last Sold Price,City,Zip,State
0,0,540 Pine Ln,3825000.0,"540 Pine Ln, Los Altos, CA 94022 is a single f...",SingleFamily,1969.0,"Heating - 2+ Zones, Central Forced Air - Gas","Multi-Zone, Central AC, Whole House / Attic Fan","Garage, Garage - Attached, Covered",1.0,...,"Garage, Garage - Attached, Covered",886486.0,12580.0,2019-10-24,4198000.0,NaN,NaN,Los Altos,94022,CA
1,1,1727 W 67th St,505000.0,"HURRY, HURRY.......Great house 3 bed and 2 bat...",SingleFamily,1926.0,Combination,"Wall/Window Unit(s), Evaporative Cooling, See ...","Detached Carport, Garage",4047.0,...,"Detached Carport, Garage",505000.0,6253.0,2019-10-16,525000.0,2019-08-30,328000.0,Los Angeles,90047,CA
2,2,28093 Pine Ave,140000.0,'THE PERFECT CABIN TO FLIP! Strawberry deligh...,SingleFamily,1958.0,Forced air,NaN,0 spaces,9147.0,...,NaN,49627.0,468.0,2019-08-25,180000.0,NaN,NaN,Strawberry,95375,CA
3,3,10750 Braddock Dr,1775000.0,Rare 2-story Gated 5 bedroom Modern Mediterran...,SingleFamily,1947.0,Central,Central Air,"Detached Carport, Driveway, Garage - Two Door",NaN,...,"Detached Carport, Driveway, Garage - Two Door",1775000.0,20787.0,2019-10-24,1895000.0,2016-08-30,1500000.0,Culver City,90230,CA
4,4,7415 O Donovan Rd,1175000.0,Beautiful 200 acre ranch land with several pas...,VacantLand,NaN,NaN,NaN,0 spaces,NaN,...,NaN,NaN,NaN,2019-06-07,1595000.0,2016-06-27,900000.0,Creston,93432,CA


In [100]:
# ---------------------- 2. 【关键】只保留数值列 ----------------------
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
target_col = "Sold Price"  # 你真实的房价列名
numeric_cols.remove(target_col)  # 移除标签列

# 只保留数值列 + 标签列
train_clean = train_df[numeric_cols + [target_col]].copy()
test_clean = test_df[numeric_cols].copy()

In [101]:
# ---------------------- 3. 查看数据 ----------------------
print("\n===== 可用数值特征 =====")
print(train_clean.columns.tolist())

print("\n===== 缺失值统计 =====")
print(train_clean.isnull().sum())


===== 可用数值特征 =====
['Id', 'Year built', 'Lot', 'Bathrooms', 'Full bathrooms', 'Total interior livable area', 'Total spaces', 'Garage spaces', 'Elementary School Score', 'Elementary School Distance', 'Middle School Score', 'Middle School Distance', 'High School Score', 'High School Distance', 'Tax assessed value', 'Annual tax amount', 'Listed Price', 'Last Sold Price', 'Zip', 'Sold Price']

===== 缺失值统计 =====
Id                                 0
Year built                      1045
Lot                            14181
Bathrooms                       3465
Full bathrooms                  7865
Total interior livable area     2526
Total spaces                     916
Garage spaces                    917
Elementary School Score         4896
Elementary School Distance      4742
Middle School Score            16705
Middle School Distance         16704
High School Score               5219
High School Distance            5001
Tax assessed value              3652
Annual tax amount               4

In [102]:
# ---------------------- 4. 填充缺失值（必须做，否则训练报错） ----------------------
train_clean = train_clean.fillna(train_clean.median())
test_clean = test_clean.fillna(test_clean.median())

In [103]:
# ---------------------- 5. 相关性分析（绝对不报错） ----------------------
print("\n===== 特征与房价相关性（从高到低）=====")
corr = train_clean.corr()[target_col].sort_values(ascending=False)
print(corr)


===== 特征与房价相关性（从高到低）=====
Sold Price                     1.000000
Annual tax amount              0.720285
Tax assessed value             0.719566
Listed Price                   0.612651
Last Sold Price                0.535457
Bathrooms                      0.496731
Full bathrooms                 0.491530
Elementary School Score        0.259437
High School Score              0.188246
Middle School Score            0.168184
Garage spaces                  0.009594
Total spaces                   0.007864
Total interior livable area   -0.001465
Lot                           -0.004796
Year built                    -0.026955
Middle School Distance        -0.035317
Zip                           -0.049961
Id                            -0.051396
Elementary School Distance    -0.078396
High School Distance          -0.090286
Name: Sold Price, dtype: float64


## 2. torch 房价预测

In [104]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


In [105]:
# 1. 特征和标签
X = train_clean.drop(columns=[target_col]).values
y = train_clean[target_col].values


In [106]:
# 2. 划分训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42) 

In [107]:
# 3. 标准化
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
test = scaler.transform(test_clean.values)

y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_val = y_scaler.transform(y_val.reshape(-1, 1))


In [108]:
# 4. 转换为 PyTorch 张量
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

test = torch.tensor(test, dtype=torch.float32)


In [109]:
# 5. 定义模型
class HousePriceModel(nn.Module):
    def __init__(self, input_dim):
        super(HousePriceModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.net(x)
        return x

In [110]:
# 6. 初始化模型、损失函数和优化器
model = HousePriceModel(X_train.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [111]:
# 7. 训练
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val)
            val_loss = criterion(val_outputs, y_val)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")

Epoch [10/100], Loss: 0.7054, Val Loss: 0.7858
Epoch [20/100], Loss: 0.5056, Val Loss: 0.5888
Epoch [30/100], Loss: 0.3936, Val Loss: 0.4683
Epoch [40/100], Loss: 0.3771, Val Loss: 0.4454
Epoch [50/100], Loss: 0.3567, Val Loss: 0.4434
Epoch [60/100], Loss: 0.3424, Val Loss: 0.4303
Epoch [70/100], Loss: 0.3262, Val Loss: 0.4138
Epoch [80/100], Loss: 0.3083, Val Loss: 0.3988
Epoch [90/100], Loss: 0.2893, Val Loss: 0.3792
Epoch [100/100], Loss: 0.2704, Val Loss: 0.3579


In [112]:
# 8. 测试集预测并生成提交文件
model.eval()
with torch.no_grad():
    test_outputs = model(test).numpy()
test_outputs = y_scaler.inverse_transform(test_outputs).flatten()
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "Sold Price": test_outputs
})

In [114]:
submission.to_csv("submission.csv", index=False)